In [1]:
# Architecture Decision Framework

In [2]:
# Objectives

# By the end of this notebook you will be able to:
# - Apply a five-axis decision model to any AI architecture problem
# - Distinguish between agent, workflow, hybrid, and traditional software approaches with quantified rationale
# - Identify when a hybrid architecture is the correct choice and articulate why
# - Produce a one-page decision matrix for a given Walmart business use case
# - Validate and export your decision matrix in a format suitable for ARB submission

# Deliverable: decision_matrix.json

In [ ]:
# Problem statement:

# For a given AI-related business problem, should we build traditional software, a fixed AI workflow, or an intelligent AI agent?

# The three architecture options
# 1. Traditional software: For example, checking whether a return is allowed based on purchase date and product category can mostly be handled through fixed business rules.

# 2. Workflow or chain:
# For example:
# a. Fetch store sales data
# b. Fetch inventory data
# c. Calculate KPIs
# d. Ask an LLM to write a summary
# e. Generate the final report
# The process is intelligent, but the steps are still controlled by us.

# 3. AI agent:
# For example, analysing supplier risk may require the system to:
# a. Read news
# b. Analyse financial reports
# c. Check geopolitical developments
# d. Compare internal delivery performance
# e. Decide which source requires deeper investigation
# f. Produce a final risk assessment
# Because the reasoning path may be different for every supplier, an agent may be justified.

In [3]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print('Environment loaded.')
print(f'OpenAI client ready: {client is not None}')

Environment loaded.
OpenAI client ready: True


## The Architecture Decision Problem

Every AI project begins with a question the team usually answers by gut feel: *should this be an agent?*
In production, gut feel costs money. A mis-classified problem -- one that gets an agent when a workflow
was sufficient -- adds latency, cost, observability complexity, and failure surface area for no benefit.

There are four architecture archetypes:

| Archetype | When to use | When NOT to use |
|-----------|-------------|------------------|
| **Traditional software** | Deterministic rules, structured input/output, no ambiguity | Tasks requiring language understanding or dynamic reasoning |
| **Workflow (chain)** | Known steps, fixed sequence, no branching on content | Tasks requiring mid-process decisions based on intermediate results |
| **Hybrid (Agent + Workflow)** | High reasoning need on some steps, hard latency or cost constraint on others | Tasks that are uniformly simple or uniformly complex |
| **Agent** | Open-ended reasoning, dynamic tool selection, multi-step planning under uncertainty | Latency-critical tasks, fully deterministic tasks, tasks where all steps are known upfront |

The decision is not about capability -- an agent can technically do anything a workflow can.
The decision is about cost, latency, predictability, and operational risk.

The hybrid archetype exists because real production systems frequently face a contradiction:
the task is complex enough to need intelligent reasoning, but the volume or latency budget prevents
running a full agent on every query. The correct response to that contradiction is not to compromise
by picking either pure option -- it is to design a system where a workflow backbone handles
predictable steps and an agent sub-component is invoked only where dynamic reasoning is required.

## The Five-Axis Decision Model

Score each axis 1-5 for your use case. Higher scores push toward agents.

| Axis | Score 1 | Score 5 |
|------|---------|----------|
| **Task Complexity** | Single step, fully deterministic | Multi-step, dynamic branching required |
| **Latency Requirement** | Real-time (< 500ms) | Batch or async acceptable (> 10s) |
| **Cost Ceiling** | Pennies per query | Dollars per query acceptable |
| **Risk Tolerance** | Zero hallucination tolerance | Some imprecision acceptable if caught downstream |
| **Update Frequency** | Logic changes rarely | Requirements change weekly |

In [4]:
AXES = ['task_complexity', 'latency_tolerance', 'cost_ceiling', 'risk_tolerance', 'update_frequency']

AXIS_DESCRIPTIONS = {
    'task_complexity':    'How many distinct decision points exist in the task?',
    'latency_tolerance':  'How much time can the system take to respond?',
    'cost_ceiling':       'What is the acceptable cost per query?',
    'risk_tolerance':     'How much imprecision is acceptable in the output?',
    'update_frequency':   'How often do requirements or logic change?',
}

THRESHOLDS = {
    'traditional': (5, 12),   # total score range for pure archetypes
    'workflow':    (13, 18),
    'agent':       (19, 25),
}

# Hybrid override constants.
# A task triggers hybrid when it demands high reasoning (task_complexity >= 4)
# but faces a hard operational constraint on latency or cost (score <= 2).
# This combination means neither a pure agent nor a pure workflow is sufficient.
HYBRID_COMPLEXITY_MIN = 4
HYBRID_CONSTRAINT_MAX = 2


# scores = {
#     'task_complexity': 4,
#     'latency_tolerance': 3,
#     'cost_ceiling': 2,
#     'risk_tolerance': 3,
#     'update_frequency': 4
# }

def recommend_architecture(scores: dict) -> dict:
    total = sum(scores.values())
    complexity = scores.get('task_complexity', 0)
    latency = scores.get('latency_tolerance', 0)
    cost = scores.get('cost_ceiling', 0)

    # Hybrid override is evaluated before the score bands.
    # When a task is both complex (needing intelligent reasoning on some steps)
    # and constrained (tight latency or cost preventing full agent deployment),
    # the score-band result would be misleading -- the correct architecture is
    # a workflow backbone with an agent sub-component on the complex steps only.
    if complexity >= HYBRID_COMPLEXITY_MIN and (latency <= HYBRID_CONSTRAINT_MAX or cost <= HYBRID_CONSTRAINT_MAX):
        return {
            'architecture': 'Hybrid (Agent + Workflow)',
            'total_score': total,
            'hybrid_override': True,
            'rationale': (
                'High task complexity coexists with a strict latency or cost constraint. '
                'Neither a pure agent nor a pure workflow fits: agent overhead is unacceptable '
                'on every step, but deterministic logic cannot handle all branches. '
                'A hybrid architecture uses a workflow backbone to route and handle predictable '
                'steps, with an agent sub-component invoked only for steps requiring dynamic reasoning.'
            ),
        }

    if total <= 12:
        arch = 'Traditional Software'
        rationale = (
            'Low complexity, strict latency, or low cost ceiling. '
            'A rule-based or deterministic pipeline is faster, cheaper, and more predictable.'
        )
    elif total <= 18:
        arch = 'Workflow (Chain)'
        rationale = (
            'Moderate complexity with known steps. '
            'A fixed-sequence workflow gives you LLM capability without agent overhead.'
        )
    else:
        arch = 'Agent'
        rationale = (
            'High complexity, dynamic branching, or rapidly changing requirements. '
            'An agent is justified -- but implement observability and cost controls from day one.'
        )
    return {'architecture': arch, 'total_score': total, 'hybrid_override': False, 'rationale': rationale}


print('Decision model loaded.')
print('Axes:', AXES)
print('Hybrid override fires when: task_complexity >=', HYBRID_COMPLEXITY_MIN,
      'AND (latency_tolerance <=', HYBRID_CONSTRAINT_MAX, 'OR cost_ceiling <=', HYBRID_CONSTRAINT_MAX, ')')

Decision model loaded.
Axes: ['task_complexity', 'latency_tolerance', 'cost_ceiling', 'risk_tolerance', 'update_frequency']
Hybrid override fires when: task_complexity >= 4 AND (latency_tolerance <= 2 OR cost_ceiling <= 2 )


In [5]:
# Walmart Use Case 1: Automated Returns Processing

# Business context: Walmart processes approximately 1 million returns per day across all channels. The current system requires a customer service associate to manually look up the purchase history, check the return policy for the product category, determine eligibility, and process the refund. The business wants to automate this for self-service kiosks and the Walmart app.

# Key facts:
# - Policy rules are documented and rarely change (quarterly updates)
# - Input is structured: item ID, purchase date, reason code, customer ID
# - 85% of cases follow one of six standard patterns
# - Latency requirement: response within 3 seconds
# - Cost sensitivity: millions of transactions per day
# - A wrong decision (approving ineligible return) has direct financial impact

In [6]:
use_case_1 = {
    'name': 'Automated Returns Processing',
    'scores': {
        'task_complexity':   2,  # mostly deterministic policy lookup
        'latency_tolerance': 1,  # 3 second hard limit
        'cost_ceiling':      1,  # millions/day -- must be sub-cent
        'risk_tolerance':    1,  # wrong approval = direct financial loss
        'update_frequency':  2,  # policy changes quarterly
    },
    'constraints': {
        'latency_p95_sec': 3,
        'cost_ceiling_per_query_usd': 0.001,
        'daily_volume': 1_000_000,
    }
}

result_1 = recommend_architecture(use_case_1['scores'])

print(f'Use Case 1: {use_case_1["name"]}')
print(f'Total Score: {result_1["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_1["hybrid_override"]}')
print(f'Recommended Architecture: {result_1["architecture"]}')
print(f'Rationale: {result_1["rationale"]}')

Use Case 1: Automated Returns Processing
Total Score: 7 / 25
Hybrid Override Applied: False
Recommended Architecture: Traditional Software
Rationale: Low complexity, strict latency, or low cost ceiling. A rule-based or deterministic pipeline is faster, cheaper, and more predictable.


## Walmart Use Case 2: Supplier Risk Intelligence

**Business context:** Walmart sources from over 100,000 suppliers globally. The procurement team
needs to continuously assess supplier risk across financial stability, geopolitical exposure,
regulatory compliance, and delivery performance. Currently, analysts manually read supplier reports,
news feeds, and financial filings to produce quarterly risk scorecards.

**Key facts:**
- Risk signals come from unstructured sources: news, filings, analyst reports, internal data
- The reasoning required varies significantly by supplier and risk type
- Analysts currently spend 3 hours per supplier per quarter
- Latency tolerance: results needed within 30 minutes of a news event
- Risk assessment logic changes as new risk categories emerge
- A missed risk signal has supply chain consequences, not immediate financial harm

In [7]:
use_case_2 = {
    'name': 'Supplier Risk Intelligence',
    'scores': {
        'task_complexity':   5,  # unstructured multi-source reasoning
        'latency_tolerance': 4,  # 30 min acceptable
        'cost_ceiling':      4,  # per-supplier cost, low volume
        'risk_tolerance':    3,  # missed signal is bad but not immediate
        'update_frequency':  4,  # risk categories evolve frequently
    },
    'constraints': {
        'latency_p95_sec': 1800,
        'cost_ceiling_per_query_usd': 2.00,
        'daily_volume': 500,
    }
}

result_2 = recommend_architecture(use_case_2['scores'])
print(f'Use Case 2: {use_case_2["name"]}')
print(f'Total Score: {result_2["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_2["hybrid_override"]}')
print(f'Recommended Architecture: {result_2["architecture"]}')
print(f'Rationale: {result_2["rationale"]}')

Use Case 2: Supplier Risk Intelligence
Total Score: 20 / 25
Hybrid Override Applied: False
Recommended Architecture: Agent
Rationale: High complexity, dynamic branching, or rapidly changing requirements. An agent is justified -- but implement observability and cost controls from day one.


## Walmart Use Case 3: Store Performance Analytics Reporter

**Business context:** District managers oversee 15-20 Walmart stores each.
Every Monday morning, they manually compile a performance report from five internal systems
(sales data, inventory, staffing, shrinkage, customer satisfaction) and write a summary
with recommended actions for store managers.

**Key facts:**
- All five source systems have structured API access
- The report format is semi-standardised but the narrative varies by week
- District managers spend 2 hours on this every Monday
- Latency tolerance: report must be ready by 8am Monday
- The KPIs tracked are stable but the recommended actions depend on context
- Wrong recommendations are reviewed before action -- human in the loop exists

In [8]:
use_case_3 = {
    'name': 'Store Performance Analytics Reporter',
    'scores': {
        'task_complexity':   3,  # structured data pull + narrative generation
        'latency_tolerance': 5,  # overnight batch acceptable
        'cost_ceiling':      3,  # moderate -- weekly per district manager
        'risk_tolerance':    3,  # human reviews before action
        'update_frequency':  3,  # KPIs stable, narrative context varies
    },
    'constraints': {
        'latency_p95_sec': 3600,
        'cost_ceiling_per_query_usd': 0.50,
        'daily_volume': 50,
    }
}

result_3 = recommend_architecture(use_case_3['scores'])
print(f'Use Case 3: {use_case_3["name"]}')
print(f'Total Score: {result_3["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_3["hybrid_override"]}')
print(f'Recommended Architecture: {result_3["architecture"]}')
print(f'Rationale: {result_3["rationale"]}')

Use Case 3: Store Performance Analytics Reporter
Total Score: 17 / 25
Hybrid Override Applied: False
Recommended Architecture: Workflow (Chain)
Rationale: Moderate complexity with known steps. A fixed-sequence workflow gives you LLM capability without agent overhead.


## Walmart Use Case 4: Customer Service Live Chat Assistant

**Business context:** Walmart handles approximately 2 million customer service interactions per day
across chat, app, and in-store kiosks. The majority of queries are simple and structured:
order status, return eligibility, store hours, and price lookups. However, approximately
15% of queries involve complex, multi-intent scenarios -- disputed charges on partial orders,
damaged-item claims requiring policy interpretation, and price match disputes with promotional
conditions. A pure workflow cannot handle the complex 15%. A pure agent is too expensive and
too slow for the simple 85% at this volume.

**Key facts:**
- Live chat requires a response within 3 seconds to avoid customer drop-off
- At 2M interactions per day, every cent of per-query cost adds USD 20,000 per day
- The complex 15% requires multi-step reasoning: retrieving order history, interpreting
  policy, and generating a personalised resolution -- not a fixed sequence
- Promotions and return policies change weekly, so the routing logic must be current
- A wrong resolution can be corrected by escalation to a human agent -- not zero-risk,
  but not catastrophic

**Why this is not a Workflow:** The complex query segment cannot be handled by a fixed
sequence of steps because the required tools and reasoning path vary per query.

**Why this is not an Agent:** Latency and cost constraints at 2M daily volume make a
full agent architecture economically and operationally infeasible for all queries.

**Why Hybrid:** A query classifier routes each incoming message. Simple queries go to a
deterministic workflow handler. Complex queries are passed to a constrained agent sub-component
with a defined tool set and a strict token budget.

In [9]:
use_case_4 = {
    'name': 'Customer Service Live Chat Assistant',
    'scores': {
        'task_complexity':   4,  # complex reasoning needed for ~15% of queries; NLU required throughout
        'latency_tolerance': 2,  # live chat -- 3-second response window
        'cost_ceiling':      2,  # 2M interactions/day -- per-query cost is operationally critical
        'risk_tolerance':    3,  # wrong answer causes escalation, not direct financial harm
        'update_frequency':  4,  # promotions and return policies change weekly
    },
    'constraints': {
        'latency_p95_sec': 3,
        'cost_ceiling_per_query_usd': 0.002,
        'daily_volume': 2_000_000,
    }
}

result_4 = recommend_architecture(use_case_4['scores'])
print(f'Use Case 4: {use_case_4["name"]}')
print(f'Total Score: {result_4["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_4["hybrid_override"]}')
print(f'Recommended Architecture: {result_4["architecture"]}')
print()
print('Note: total score 15 would place this in Workflow (Chain) by score band alone.')
print('The hybrid override fires because task_complexity=4 and latency_tolerance=2.')
print(f'Rationale: {result_4["rationale"]}')

Use Case 4: Customer Service Live Chat Assistant
Total Score: 15 / 25
Hybrid Override Applied: True
Recommended Architecture: Hybrid (Agent + Workflow)

Note: total score 15 would place this in Workflow (Chain) by score band alone.
The hybrid override fires because task_complexity=4 and latency_tolerance=2.
Rationale: High task complexity coexists with a strict latency or cost constraint. Neither a pure agent nor a pure workflow fits: agent overhead is unacceptable on every step, but deterministic logic cannot handle all branches. A hybrid architecture uses a workflow backbone to route and handle predictable steps, with an agent sub-component invoked only for steps requiring dynamic reasoning.


In [10]:
use_cases = [
    (use_case_1, result_1),
    (use_case_2, result_2),
    (use_case_3, result_3),
    (use_case_4, result_4),
]

print(f"{'Use Case':<45} {'Score':>6} {'Override':>8}  {'Recommended Architecture':<30}")
print('-' * 92)
for uc, res in use_cases:
    override_flag = 'YES' if res['hybrid_override'] else 'no'
    print(f'{uc["name"]:<45} {res["total_score"]:>6} {override_flag:>8}  {res["architecture"]:<30}')

print()
print('Key insight: Use Case 4 scores 15/25 -- identical band to Use Case 3.')
print('The hybrid override distinguishes them: Use Case 4 has task_complexity=4')
print('combined with latency_tolerance=2 and cost_ceiling=2.')
print('Score-band alone would incorrectly recommend Workflow for both.')

Use Case                                       Score Override  Recommended Architecture      
--------------------------------------------------------------------------------------------
Automated Returns Processing                       7       no  Traditional Software          
Supplier Risk Intelligence                        20       no  Agent                         
Store Performance Analytics Reporter              17       no  Workflow (Chain)              
Customer Service Live Chat Assistant              15      YES  Hybrid (Agent + Workflow)     

Key insight: Use Case 4 scores 15/25 -- identical band to Use Case 3.
The hybrid override distinguishes them: Use Case 4 has task_complexity=4
combined with latency_tolerance=2 and cost_ceiling=2.
Score-band alone would incorrectly recommend Workflow for both.


## LLM-Assisted Decision Justification

For ARB submission, the recommendation must be defensible in prose, not just a score.
The function below calls GPT-4o-mini to generate a one-paragraph ARB-ready justification
for each use case, grounded in the scored axes.

For hybrid use cases, the justification prompt is extended to require the model to explain
why neither pure alternative is acceptable and how the workflow backbone and agent
sub-component divide responsibility.

In [11]:
def generate_arb_justification(use_case: dict, recommendation: dict) -> str:
    axes_text = '\n'.join([
        f'  {axis}: {score}/5 -- {AXIS_DESCRIPTIONS[axis]}'
        for axis, score in use_case['scores'].items()
    ])
# task_complexity: 2/5 -- How many distinct decision points exist in the task?
# latency_tolerance: 1/5 -- How much time can the system take to respond?
# cost_ceiling: 1/5 -- What is the acceptable cost per query?
# risk_tolerance: 1/5 -- How much imprecision is acceptable in the output?
# update_frequency: 2/5 -- How often do requirements or logic change?

    if recommendation.get('hybrid_override'):
        hybrid_instruction = (
            '5. Explain why a pure workflow is insufficient for the complex query segment.\n'
            '6. Explain why a pure agent is infeasible given the latency and cost constraints.\n'
            '7. Describe how the workflow backbone and agent sub-component divide responsibility.'
        )
        extra_note = (
            f'\n\nNote: This recommendation is the result of a hybrid override. '
            f'The total score of {recommendation["total_score"]} would place this use case in the '
            f'Workflow (Chain) band, but the combination of task_complexity >= 4 and a hard '
            f'operational constraint on latency or cost makes a pure workflow insufficient.'
        )
    else:
        hybrid_instruction = '5. Is written in formal ARB language -- no hedging, no bullet points'
        extra_note = ''

    prompt = (
        f'You are a senior AI architect writing a formal architecture decision note for a Walmart ARB review.\n\n'
        f'Use case: {use_case["name"]}\n'
        f'Recommended architecture: {recommendation["architecture"]}\n'
        f'Total decision score: {recommendation["total_score"]} / 25\n'
        f'{extra_note}\n\n'
        f'Axis scores:\n{axes_text}\n\n'
        f'Write a single concise paragraph (4-6 sentences) that:\n'
        f'1. States the recommended architecture and why\n'
        f'2. Calls out the most influential axis scores\n'
        f'3. Names the primary risk of the alternative architectures\n'
        f'4. Is written in formal ARB language -- no hedging, no bullet points\n'
        f'{hybrid_instruction}'
    )
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3,
        max_tokens=350,
    )
    return response.choices[0].message.content.strip()


print('Generating ARB justifications...')
for uc, res in use_cases:
    justification = generate_arb_justification(uc, res)
    uc['arb_justification'] = justification
    print(f'\n--- {uc["name"]} ---')
    print(justification)

Generating ARB justifications...

--- Automated Returns Processing ---
The recommended architecture for the Automated Returns Processing use case is Traditional Software, primarily due to its suitability for handling the task complexity and update frequency inherent in the process. The decision score reflects significant challenges in latency tolerance, cost ceiling, and risk tolerance, with scores of 1/5 indicating a low tolerance for delays, expenses, and imprecision. The most influential axis scores are task complexity at 2/5 and update frequency at 2/5, suggesting a moderate level of decision points and infrequent changes in requirements. The primary risk associated with alternative architectures, such as microservices or serverless solutions, lies in their potential inability to meet the stringent requirements for response time and cost-effectiveness, which could lead to operational inefficiencies and increased overhead. Therefore, Traditional Software is deemed the most viable op

## Full Decision Matrix Builder

The complete decision matrix includes the use case context, axis scores, recommendation,
cost projections, and ARB justification. This is the format expected at architecture review.

For hybrid use cases, the matrix includes a `hybrid_override_applied` flag and an extended
`alternatives_considered` list covering all four archetypes.

In [13]:
ALL_ARCHETYPES = [
    'Traditional Software',
    'Workflow (Chain)',
    'Hybrid (Agent + Workflow)',
    'Agent',
]


def build_decision_matrix(use_case: dict, recommendation: dict) -> dict:
    constraints = use_case.get('constraints', {})
    daily_volume = constraints.get('daily_volume', 0)
    cost_per_query = constraints.get('cost_ceiling_per_query_usd', 0)
    monthly_cost_ceiling = daily_volume * cost_per_query * 30

    return {
        'use_case': use_case['name'],
        'axis_scores': use_case['scores'],
        'total_score': recommendation['total_score'],
        'recommended_architecture': recommendation['architecture'],
        'hybrid_override_applied': recommendation.get('hybrid_override', False),
        'rationale': recommendation['rationale'],
        'arb_justification': use_case.get('arb_justification', ''),
        'constraints': {
            'latency_p95_sec': constraints.get('latency_p95_sec'),
            'cost_ceiling_per_query_usd': cost_per_query,
            'daily_volume': daily_volume,
            'monthly_cost_ceiling_usd': round(monthly_cost_ceiling, 2),
        },
        'alternatives_considered': [
            a for a in ALL_ARCHETYPES
            if a != recommendation['architecture']
        ],
        'decision_status': 'DRAFT',
    }


matrices = []
for uc, res in use_cases:
    matrix = build_decision_matrix(uc, res)
    matrices.append(matrix)
    print(f'Built matrix for: {matrix["use_case"]}')
    print(f'  Architecture: {matrix["recommended_architecture"]}')
    print(f'  Hybrid override: {matrix["hybrid_override_applied"]}')
    print(f'  Monthly cost ceiling: ${matrix["constraints"]["monthly_cost_ceiling_usd"]:,.2f}')

Built matrix for: Automated Returns Processing
  Architecture: Traditional Software
  Hybrid override: False
  Monthly cost ceiling: $30,000.00
Built matrix for: Supplier Risk Intelligence
  Architecture: Agent
  Hybrid override: False
  Monthly cost ceiling: $30,000.00
Built matrix for: Store Performance Analytics Reporter
  Architecture: Workflow (Chain)
  Hybrid override: False
  Monthly cost ceiling: $750.00
Built matrix for: Customer Service Live Chat Assistant
  Architecture: Hybrid (Agent + Workflow)
  Hybrid override: True
  Monthly cost ceiling: $120,000.00


## Lab Exercise: Design Your Own Matrix

Apply the five-axis model to a new Walmart use case of your choice.
Suggestions: demand forecasting assistant, HR policy Q&A, store layout optimiser,
vendor onboarding guide, price matching automation, or pharmacy prescription assistant.

Pay attention to whether your use case triggers the hybrid override. If it does, articulate
in your ARB justification how you would split the query routing between the workflow backbone
and the agent sub-component.

Fill in the dictionary below and run the cell.

In [14]:
my_use_case = {
    'name': 'YOUR USE CASE NAME HERE',
    'scores': {
        'task_complexity':   3,  # replace with your score 1-5
        'latency_tolerance': 3,  # replace with your score 1-5
        'cost_ceiling':      3,  # replace with your score 1-5
        'risk_tolerance':    3,  # replace with your score 1-5
        'update_frequency':  3,  # replace with your score 1-5
    },
    'constraints': {
        'latency_p95_sec': 5,
        'cost_ceiling_per_query_usd': 0.01,
        'daily_volume': 1000,
    }
}

my_result = recommend_architecture(my_use_case['scores'])
my_justification = generate_arb_justification(my_use_case, my_result)
my_use_case['arb_justification'] = my_justification
my_matrix = build_decision_matrix(my_use_case, my_result)
matrices.append(my_matrix)

print(f'Your use case: {my_use_case["name"]}')
print(f'Score: {my_result["total_score"]} / 25')
print(f'Hybrid override applied: {my_result["hybrid_override"]}')
print(f'Recommendation: {my_result["architecture"]}')
print()
print('ARB Justification:')
print(my_justification)

Your use case: YOUR USE CASE NAME HERE
Score: 15 / 25
Hybrid override applied: False
Recommendation: Workflow (Chain)

ARB Justification:
After careful consideration, the recommended architecture for the use case "YOUR USE CASE NAME HERE" is a Workflow (Chain) architecture, as it effectively accommodates the task complexity and moderate latency tolerance inherent in the requirements. The axis scores indicate a balanced need for decision-making flexibility, with notable scores of 3/5 across task complexity, latency tolerance, cost ceiling, risk tolerance, and update frequency, suggesting that the system must handle multiple decision points while remaining cost-effective and adaptable to changes. The primary risk associated with alternative architectures, such as a monolithic approach, lies in their potential inability to efficiently manage the dynamic nature of the task, leading to increased latency and reduced responsiveness to evolving requirements. Therefore, the Workflow (Chain) arc

## Validate and Export Decision Matrix

The validation function checks that every required field is complete and
no axis score is left at the default. This mirrors the completeness gate
used in the Module 5 ARB capstone.

In [15]:
def validate_decision_matrix(matrices: list) -> dict:
    results = []
    all_valid = True
    for m in matrices:
        issues = []
        if m['use_case'] in ('', 'YOUR USE CASE NAME HERE'):
            issues.append('Use case name is placeholder')
        for axis in AXES:
            score = m['axis_scores'].get(axis, 0)
            if not (1 <= score <= 5):
                issues.append(f'Axis {axis} score {score} is out of range 1-5')
        if not m.get('arb_justification'):
            issues.append('ARB justification is missing')
        status = 'PASS' if not issues else 'FAIL'
        if status == 'FAIL':
            all_valid = False
        results.append({'use_case': m['use_case'], 'status': status, 'issues': issues})
    return {'overall': 'PASS' if all_valid else 'FAIL', 'results': results}


validation = validate_decision_matrix(matrices)
print(f'Validation result: {validation["overall"]}')
for r in validation['results']:
    status_label = 'PASS' if r['status'] == 'PASS' else 'FAIL'
    print(f'  {status_label}  {r["use_case"]}')
    for issue in r['issues']:
        print(f'    - {issue}')

output = {
    'programme': 'Advanced Agentic AI -- Production Engineering',
    'track': 'India',
    'module': 'Module 1 -- Engineering Decisions for AI Systems',
    'notebook': 'IN01a',
    'validation': validation,
    'decision_matrices': matrices,
}

out_path = 'decision_matrix.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\nDecision matrix saved to {out_path}')
print(f'Total use cases documented: {len(matrices)}')

Validation result: FAIL
  PASS  Automated Returns Processing
  PASS  Supplier Risk Intelligence
  PASS  Store Performance Analytics Reporter
  PASS  Customer Service Live Chat Assistant
  FAIL  YOUR USE CASE NAME HERE
    - Use case name is placeholder

Decision matrix saved to decision_matrix.json
Total use cases documented: 5


## Summary

You have applied the five-axis decision model to four Walmart use cases, including one that
triggers the hybrid override, and produced an ARB-ready decision matrix for each.

**Key takeaways:**

- The architecture decision is a function of volume, latency, cost, and risk -- not capability
- An agent is not always the best answer; over-engineering has real production costs
- The score band captures 80% of cases well -- the hybrid override covers the remaining gap
- Hybrid is not a compromise: it is the correct design when requirements are genuinely mixed
- The hybrid override condition (complexity >= 4 AND constraint <= 2) is intentionally narrow;
  it fires only when both signals are present, not when one is borderline
- The decision matrix forces you to quantify assumptions that are usually left implicit
- The ARB justification paragraph is what you will read aloud at the review board

**Your deliverable:** `decision_matrix.json`

**Next:** IN02 -- RAG vs Fine-tuning vs Prompting: Quantified Decision Framework and LoRA Lab